# L12 — Multi-Server Queue: M/M/c and the Erlang-C Formula

**Module**: M05 | **Chapter**: 7 | **Lecture**: L12

## Learning Objectives
By the end of this notebook you will be able to:
1. Explain why adding servers has diminishing returns as utilisation approaches 1.
2. Implement M/M/c in SimPy using `Resource(env, capacity=c)`.
3. Compute the Erlang-C formula and verify it against simulation output.
4. Quantify the benefit of server pooling vs. dedicated queues.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import factorial
from scipy.stats import t as t_dist

## 1. M/M/c Analytical Formulas

For M/M/c: Poisson(λ) arrivals, Exp(μ) service, *c* identical servers, FCFS discipline.

Per-server utilisation: ρ = λ/(cμ) < 1

**Erlang-C formula** — probability an arriving customer must wait:
$$C(c, a) = \frac{\frac{a^c}{c!} \cdot \frac{1}{1-\rho}}{\sum_{n=0}^{c-1}\frac{a^n}{n!} + \frac{a^c}{c!}\cdot\frac{1}{1-\rho}}$$
where $a = \lambda/\mu$ (traffic intensity).

Mean wait in queue: $W_q = C(c,a) / (c\mu - \lambda)$

In [ ]:
def erlang_c(c: int, lam: float, mu: float) -> dict:
    """Compute M/M/c analytical performance measures."""
    rho = lam / (c * mu)
    assert rho < 1, f"System unstable: ρ={rho:.3f} >= 1"
    a = lam / mu  # traffic intensity

    # Erlang-C numerator and denominator
    sum_terms = sum(a**n / factorial(n) for n in range(c))
    last = (a**c / factorial(c)) / (1 - rho)
    C = last / (sum_terms + last)  # P(wait > 0)

    Wq  = C / (c * mu - lam)
    W   = Wq + 1.0 / mu
    Lq  = lam * Wq
    L   = lam * W
    return {'C': C, 'rho': rho, 'Wq': Wq, 'W': W, 'Lq': Lq, 'L': L}


# Compare M/M/1, M/M/2, M/M/3 at same total load
lam = 6.0  # arrivals/hour
mu  = 4.0  # service rate per server

print(f"λ={lam}, μ={mu} per server")
print(f"{'c':>4s}  {'ρ':>6s}  {'C(c,a)':>8s}  {'Wq':>8s}  {'W':>8s}  {'L':>8s}")
print('-' * 50)
for c in [2, 3, 4]:
    r = erlang_c(c, lam, mu)
    print(f"{c:>4d}  {r['rho']:>6.3f}  {r['C']:>8.4f}  "
          f"{r['Wq']:>8.4f}  {r['W']:>8.4f}  {r['L']:>8.4f}")

## 2. SimPy M/M/c Implementation

Only one line changes from M/M/1: `capacity=c` in `simpy.Resource`.

In [ ]:
def mmc_simulation(lam: float, mu: float, c: int,
                   sim_time: float = 100_000, seed: int = 0) -> dict:
    """Simulate M/M/c queue and return summary statistics."""
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    server = simpy.Resource(env, capacity=c)
    waits, sojourns = [], []

    def customer():
        arrival = env.now
        with server.request() as req:
            yield req
            wait = env.now - arrival
            svc = rng.exponential(1.0 / mu)
            yield env.timeout(svc)
        waits.append(wait)
        sojourns.append(wait + svc)

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            env.process(customer())

    env.process(arrivals())
    env.run(until=sim_time)

    Wq = np.mean(waits)
    W  = np.mean(sojourns)
    n  = len(waits)
    return {'Wq': Wq, 'W': W, 'Lq': lam*Wq, 'L': lam*W,
            'rho': sum(w+s for w,s in zip(waits,sojourns)) / (c * sim_time),
            'n_served': n,
            'P_wait': sum(w > 0 for w in waits) / n}


# Verify against Erlang-C
print(f"M/M/c verification (λ={lam}, μ={mu}, T=100,000)")
print(f"{'c':>3s}  {'metric':>6s}  {'sim':>9s}  {'theory':>9s}  {'err %':>7s}")
print('-' * 45)
for c in [2, 3, 4]:
    sim    = mmc_simulation(lam, mu, c, seed=42)
    theory = erlang_c(c, lam, mu)
    for key in ['Wq', 'W']:
        err = abs(sim[key] - theory[key]) / theory[key] * 100
        print(f"{c:>3d}  {key:>6s}  {sim[key]:>9.4f}  {theory[key]:>9.4f}  {err:>6.2f}%")

## 3. The Diminishing-Returns Curve

At high utilisation, doubling servers does not halve waiting time. The relationship is highly nonlinear.

In [ ]:
# Fix total offered load a = lambda/mu, vary c
a_values = [1.5, 2.5, 3.5]  # traffic intensities
mu_fixed  = 1.0
c_range   = range(2, 8)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2166ac', '#d6604d', '#4dac26']

for col, a in zip(colors, a_values):
    lam = a * mu_fixed
    wq_vals = []
    valid_c = []
    for c in c_range:
        rho = lam / (c * mu_fixed)
        if rho < 1.0:
            r = erlang_c(c, lam, mu_fixed)
            wq_vals.append(r['Wq'] * mu_fixed)  # normalise by mu
            valid_c.append(c)
    ax.plot(valid_c, wq_vals, 'o-', color=col, lw=2,
            label=f'a=λ/μ={a}  (min c={valid_c[0]})')

ax.set_xlabel('Number of servers c')
ax.set_ylabel('Normalised wait μWq')
ax.set_title('Diminishing returns: adding servers at high load (M/M/c theory)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Server Pooling vs. Dedicated Queues

**Question**: Is one shared pool of *c* servers better than *c* separate single-server queues (each handling λ/c arrivals)?

**Answer**: Always. Pooling reduces variance and allows load balancing.

In [ ]:
def dedicated_queues_sim(lam: float, mu: float, c: int,
                         sim_time: float = 100_000, seed: int = 0) -> dict:
    """Simulate c independent M/M/1 queues each receiving lam/c arrivals."""
    lam_each = lam / c
    all_waits = []
    for k in range(c):
        rng = np.random.default_rng(seed + k)
        env = simpy.Environment()
        server = simpy.Resource(env, capacity=1)
        waits_k = []

        def customer(w_list=waits_k):
            arr = env.now
            with server.request() as req:
                yield req
                w_list.append(env.now - arr)
                yield env.timeout(rng.exponential(1.0 / mu))

        def arrivals(w_list=waits_k):
            while True:
                yield env.timeout(rng.exponential(1.0 / lam_each))
                env.process(customer(w_list))

        env.process(arrivals())
        env.run(until=sim_time)
        all_waits.extend(waits_k)

    Wq = np.mean(all_waits)
    return {'Wq': Wq, 'W': Wq + 1/mu, 'n_served': len(all_waits)}


# Compare for c=2 servers at rho=0.75
lam_test, mu_test, c_test = 6.0, 4.0, 2

pooled    = mmc_simulation(lam_test, mu_test, c_test, seed=0)
dedicated = dedicated_queues_sim(lam_test, mu_test, c_test, seed=0)
theory    = erlang_c(c_test, lam_test, mu_test)

print(f"Server pooling advantage (c={c_test}, ρ={lam_test/(c_test*mu_test):.2f}):")
print(f"                Wq (theory)  Wq (pooled sim)  Wq (dedicated)")
print(f"  M/M/{c_test} pool:  {theory['Wq']:>8.4f}       {pooled['Wq']:>8.4f}      —")
print(f"  Dedicated M/M/1:   {lam_test/(c_test*mu_test)/(mu_test*(1-lam_test/(c_test*mu_test))):>8.4f}          —      {dedicated['Wq']:>8.4f}")
print(f"  Pooling reduces Wq by ~{(dedicated['Wq']-pooled['Wq'])/dedicated['Wq']*100:.1f}%")

## 5. Staffing Analysis: Minimum Servers for a Service-Level Target

In [ ]:
# Find minimum c such that P(wait > T_target) <= target_prob
# P(wait > t | wait > 0) = exp(-(cμ-λ)t)  for M/M/c
# P(wait > t) = C(c,a) * exp(-(cμ-λ)t)

def prob_wait_exceeds(c: int, lam: float, mu: float, t: float) -> float:
    r = erlang_c(c, lam, mu)
    return r['C'] * np.exp(-(c * mu - lam) * t)


# Clinic staffing problem
# λ=8 patients/hour, μ=6 patients/hour per nurse
# Target: P(wait > 15 min) <= 0.10
lam_clinic, mu_clinic = 8.0, 6.0
T_target = 15.0 / 60.0   # 15 min in hours
target   = 0.10

print(f"Clinic: λ={lam_clinic}/hr, μ={mu_clinic}/hr/nurse")
print(f"Target: P(wait > {T_target*60:.0f} min) ≤ {target}")
print()
print(f"{'c':>3s}  {'ρ':>6s}  {'C(c,a)':>8s}  {'P(W>15min)':>12s}  {'Meets target?':>14s}")
print('-' * 55)
for c in range(2, 8):
    rho = lam_clinic / (c * mu_clinic)
    if rho >= 1.0:
        print(f"{c:>3d}  {rho:>6.3f}  {'unstable':>8s}")
        continue
    C = erlang_c(c, lam_clinic, mu_clinic)['C']
    pw = prob_wait_exceeds(c, lam_clinic, mu_clinic, T_target)
    ok = '✓' if pw <= target else '✗'
    print(f"{c:>3d}  {rho:>6.3f}  {C:>8.4f}  {pw:>12.4f}  {ok:>14s}")

---
## Try It Yourself

1. **Erlang-B (loss system)**: An M/M/c/c queue has no waiting room — customers who find all servers busy are turned away (lost). Implement the Erlang-B formula `B(c, a) = (a^c/c!) / sum(a^n/n! for n=0..c)` and simulate an M/M/3/3 queue. What fraction of customers are lost at ρ=0.8?

2. **Mixed service times**: Replace exponential service with triangular(low=2, high=10, mode=4) minutes. Simulate an M/G/c queue with c=3 servers, λ=10/hr. Compute Wq from 30 replications of T=8hr and compare to the M/M/c Erlang-C result for the same mean service time. What does the difference reveal about service-time variability?

3. **Time-varying staffing**: The clinic has λ(t) = 8 in hours 0–4, λ(t) = 12 in hours 4–8. Staff 3 nurses for the first shift and 4 for the second. Simulate and report P(wait > 15 min) for each shift separately. Is the service target met in both periods?